# 99 Indonesian Speaker Corpus — ASR Evaluation

This notebook evaluates **Whisper-small** on the 99-speaker Indonesian speech corpus.

For each speaker, the notebook:
1. Loads the full speaker recording and its reference transcript.
2. Transcribes the recording using Indonesian language settings.
3. Applies the same text normalization to reference and hypothesis.
4. Calculates speaker-level WER and CER, plus word-level substitutions, deletions, insertions, and hits.
5. Saves a CSV checkpoint after each completed speaker.

**Important:** The notebook follows the evaluation procedure used in the study. It does not perform model fine-tuning. The reference transcript is the corpus transcript file.

## 1. Environment and imports

Run this notebook in an environment with access to the corpus files and a CUDA GPU if available. CPU execution is supported but may be considerably slower.

In [ ]:
# Install dependencies if they are not already available.
# In Kaggle, enable Internet or attach packages as needed.
%pip install -q openai-whisper jiwer soundfile pandas

In [ ]:
import gc
import re
import time
from pathlib import Path

import pandas as pd
import soundfile as sf
import torch
import whisper
from jiwer import wer, cer, process_words

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

## 2. Dataset and output configuration

Set `DATASET_ROOT` to the directory containing files named `speaker01.wav` … `speaker99.wav` and `transcript01.txt` … `transcript99.txt`.

The default path reflects the Kaggle dataset path used in the original experiment. Change it if your dataset is mounted elsewhere.

In [ ]:
DATASET_ROOT = Path(
    "/kaggle/input/datasets/mouchu/indonesian-speech-corpus-for-speaker-recognition"
)

OUTPUT_DIR = Path("/kaggle/working")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

CHECKPOINT_PATH = OUTPUT_DIR / "whisper_small_99speaker_results.csv"

MODEL_NAME = "small"
LANGUAGE = "id"
TASK = "transcribe"

print("Dataset directory exists:", DATASET_ROOT.exists())
print("Checkpoint:", CHECKPOINT_PATH)

## 3. Load Whisper-small

The original evaluation used the `small` Whisper checkpoint, with Indonesian explicitly specified during transcription. FP16 is enabled when CUDA is available.

In [ ]:
whisper_model = whisper.load_model(MODEL_NAME)
print(f"Whisper-{MODEL_NAME} loaded.")

## 4. Text normalization

The same normalization function is applied to reference and hypothesis:
- convert to lowercase;
- replace punctuation/non-word characters with spaces;
- collapse repeated whitespace;
- trim leading and trailing whitespace.

Digits are retained by the `\\w` pattern.

In [ ]:
def normalize_text(text: str) -> str:
    text = text.lower()
    text = re.sub(r"[^\w\s]", " ", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()

## 5. Resume support

The notebook saves results after each speaker. If the notebook stops, rerun it and completed speaker IDs in the checkpoint will be skipped.

Delete the checkpoint CSV only if you intentionally want to start a new evaluation from scratch.

In [ ]:
RESULT_COLUMNS = [
    "speaker", "duration_sec", "reference_words", "hypothesis_words",
    "WER", "CER", "substitutions", "deletions", "insertions", "hits"
]

if CHECKPOINT_PATH.exists():
    results_df = pd.read_csv(CHECKPOINT_PATH)
    completed_speakers = set(results_df["speaker"].astype(str))
    print(f"Existing checkpoint: {len(completed_speakers)} speakers")
else:
    results_df = pd.DataFrame(columns=RESULT_COLUMNS)
    completed_speakers = set()
    print("No checkpoint found. Starting from speaker01.")

## 6. Evaluate all 99 speakers

Each full speaker WAV file is passed to Whisper. No additional audio resampling or segmentation is performed in this evaluation loop; Whisper handles its own input audio processing.

In [ ]:
for speaker_num in range(1, 100):
    speaker_id = f"speaker{speaker_num:02d}"

    if speaker_id in completed_speakers:
        print(f"[SKIP] {speaker_id} already evaluated.")
        continue

    audio_path = DATASET_ROOT / f"{speaker_id}.wav"
    transcript_path = DATASET_ROOT / f"transcript{speaker_num:02d}.txt"

    if not audio_path.is_file():
        print(f"[MISSING AUDIO] {audio_path}")
        continue
    if not transcript_path.is_file():
        print(f"[MISSING TRANSCRIPT] {transcript_path}")
        continue

    start_time = time.time()
    print(f"\nEvaluating {speaker_id} ...")

    try:
        reference = transcript_path.read_text(encoding="utf-8")
        reference_norm = normalize_text(reference)

        result = whisper_model.transcribe(
            str(audio_path),
            language=LANGUAGE,
            task=TASK,
            fp16=torch.cuda.is_available()
        )
        hypothesis = result["text"]
        hypothesis_norm = normalize_text(hypothesis)

        alignment = process_words(reference_norm, hypothesis_norm)
        speaker_wer = wer(reference_norm, hypothesis_norm)
        speaker_cer = cer(reference_norm, hypothesis_norm)
        duration_sec = sf.info(str(audio_path)).duration

        row = {
            "speaker": speaker_id,
            "duration_sec": duration_sec,
            "reference_words": len(reference_norm.split()),
            "hypothesis_words": len(hypothesis_norm.split()),
            "WER": speaker_wer * 100,
            "CER": speaker_cer * 100,
            "substitutions": alignment.substitutions,
            "deletions": alignment.deletions,
            "insertions": alignment.insertions,
            "hits": alignment.hits
        }

        results_df = pd.concat(
            [results_df, pd.DataFrame([row])],
            ignore_index=True
        )
        results_df.to_csv(CHECKPOINT_PATH, index=False)
        completed_speakers.add(speaker_id)

        print(
            f"{speaker_id}: WER={row['WER']:.2f}% | "
            f"CER={row['CER']:.2f}% | "
            f"elapsed={(time.time() - start_time)/60:.2f} min"
        )

    except Exception as exc:
        print(f"[ERROR] {speaker_id}: {type(exc).__name__}: {exc}")
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        continue

    del result, hypothesis, reference
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

print("\nEvaluation loop finished.")
print("Checkpoint saved at:", CHECKPOINT_PATH)

## 7. Speaker-level summary

The mean, median, and standard deviation below are calculated across the available speaker-level WER and CER values. They are not aggregate corpus-level WER/CER.

In [ ]:
results_df = pd.read_csv(CHECKPOINT_PATH)
results_df = results_df.sort_values("speaker").reset_index(drop=True)

print("Speakers evaluated:", len(results_df))
display(results_df)

summary = results_df[["WER", "CER"]].agg(["mean", "median", "std"]).T
display(summary)

## 8. Aggregate word-error breakdown

Aggregate WER is calculated from the total substitutions, deletions, and insertions divided by the total number of reference words. This differs from the mean of the 99 speaker-level WER values.

In [ ]:
total_ref = int(results_df["reference_words"].sum())
total_hits = int(results_df["hits"].sum())
total_S = int(results_df["substitutions"].sum())
total_D = int(results_df["deletions"].sum())
total_I = int(results_df["insertions"].sum())
total_errors = total_S + total_D + total_I

aggregate_wer = total_errors / total_ref if total_ref else float("nan")

aggregate_breakdown = pd.DataFrame({
    "Metric": ["Reference words", "Hits", "Substitutions", "Deletions",
               "Insertions", "Total errors", "Aggregate WER (%)"],
    "Value": [total_ref, total_hits, total_S, total_D, total_I,
              total_errors, aggregate_wer * 100]
})
display(aggregate_breakdown)

if total_ref:
    print(f"Substitution rate: {total_S / total_ref * 100:.2f}%")
    print(f"Deletion rate:     {total_D / total_ref * 100:.2f}%")
    print(f"Insertion rate:    {total_I / total_ref * 100:.2f}%")

if total_errors:
    print(f"Substitution share of errors: {total_S / total_errors * 100:.2f}%")
    print(f"Deletion share of errors:     {total_D / total_errors * 100:.2f}%")
    print(f"Insertion share of errors:    {total_I / total_errors * 100:.2f}%")

## 9. Export results

The per-speaker results and aggregate summary can be retained as CSV files for subsequent analysis.

In [ ]:
PER_SPEAKER_EXPORT = OUTPUT_DIR / "whisper_small_speaker_level_results.csv"
SUMMARY_EXPORT = OUTPUT_DIR / "whisper_small_summary.csv"

results_df.to_csv(PER_SPEAKER_EXPORT, index=False)
summary.to_csv(SUMMARY_EXPORT)

print("Saved:", PER_SPEAKER_EXPORT)
print("Saved:", SUMMARY_EXPORT)